# PyTorch ATen Hardware Support Matrix Exporter

This notebook is designed to run in **Google Colab**. It automates installing multiple versions of PyTorch, extracting the ATen hardware support matrix, and saving the compressed outputs directly to your Google Drive.

### Workflow Overview:
1. **Mount Google Drive**: Establish the connection to save data.
2. **Generate Exporter Script**: Write the optimized script to `/content/exporter.py` on the local file system. This script will directly construct the compressed matrix and update the registry mapping `versions.json`.
3. **Run Version Workflows**: For each PyTorch version you want to export, run the corresponding group of cells (Install -> Restart Runtime -> Run Exporter).

## Section 1: Introduction & Drive Mount

Mount Google Drive to `/content/drive` and initialize the target directory `/content/drive/MyDrive/pytorch_aten_exports` for exports.

In [ ]:
from google.colab import drive
import os

# Mount Google Drive to /content/drive
drive.mount('/content/drive')

# Setup and initialize target output directory
drive_out_dir = '/content/drive/MyDrive/pytorch_aten_exports'
os.makedirs(drive_out_dir, exist_ok=True)
print(f"Target directory initialized at: {drive_out_dir}")

## Section 2: PyTorch Installation Selector & Commands Index

Here is an installation reference table for different PyTorch backends. Use these commands to target specific hardware support arrays:

| Backend | Installation Command |
|---|---|
| **CPU** | `!pip install -q torch --index-url https://download.pytorch.org/whl/cpu` |
| **CUDA 12.1** | `!pip install -q torch --index-url https://download.pytorch.org/whl/cu121` |
| **CUDA 12.4** | `!pip install -q torch --index-url https://download.pytorch.org/whl/cu124` |
| **ROCm 6.0 (AMD)** | `!pip install -q torch --index-url https://download.pytorch.org/whl/rocm6.0` |
| **Intel XPU (Intel GPU)** | `!pip install -q torch --index-url https://download.pytorch.org/whl/xpu` |
| **Google XLA (TPU)** | `!pip install -q torch torch_xla[tpu] -f https://storage.googleapis.com/libtpu-wheels/index.html` |

## Section 3: %%writefile /content/exporter.py

Run the cell below to write the core exporter script to `/content/exporter.py`. This script queries the PyTorch dispatch table, applies regex classification via `LINE_RE` and `classify_line`, generates the compressed JSON matrix (where statuses are mapped to `'n'`, `'c'`, `'f'`, or `'o'`), and updates `versions.json` directly in Google Drive.

In [ ]:
%%writefile /content/exporter.py
import os
import re
import json
import torch

# Common hardware / backend keys to display on the blog
HARDWARE_KEYS = [
    "CPU", "CUDA", "HIP", "MPS", "XPU", "XLA", "MTIA", "MAIA", "HPU",
    "Vulkan", "Metal", "Lazy", "VE", "FPGA", "PrivateUse1", "PrivateUse2", "PrivateUse3"
]

# Regex to match dispatcher table dump lines
LINE_RE = re.compile(r"^(?P<key>[^:]+): (?P<body>.*) \[(?P<kind>[^\\]]+)\]$")

def classify_line(body: str, kind: str) -> str:
    """
    Classify dispatcher entry into a compact status for blog display.
    """
    fallthrough = body.startswith("fallthrough ")
    if kind == "autograd kernel":
        return "autograd"
    if kind in ("math kernel", "default backend kernel"):
        return "composite"
    if kind == "backend fallback":
        return "fallback"
    if fallthrough:
        return "fallback"
    if kind == "kernel":
        return "native"
    return "other"

def parse_table_for_backends(op_name, table, active_backends):
    keys = {}
    for line in table.splitlines():
        line = line.strip()
        if not line:
            continue
        m = LINE_RE.match(line)
        if not m:
            continue
        key = m.group("key").strip()
        if key in active_backends:
            body = m.group("body").strip()
            kind = m.group("kind").strip()
            status = classify_line(body, kind)
            
            short_status = "o"
            if status == "native": short_status = "n"
            elif status == "composite": short_status = "c"
            elif status == "fallback": short_status = "f"
            
            if short_status != "o":
                keys[key] = short_status
    return keys

def main():
    torch_version = torch.__version__
    print(f"Detected PyTorch version: {torch_version}")
    safe_version_name = "v" + torch_version.replace("+", "-")
    
    # 1. Dynamic backend detection
    lower_version = torch_version.lower()
    target_backend = "CPU"
    if "xpu" in lower_version or getattr(torch, "xpu", None) and torch.xpu.is_available():
        target_backend = "XPU"
    elif "cu" in lower_version or torch.cuda.is_available():
        target_backend = "CUDA"
    elif "rocm" in lower_version or "hip" in lower_version or getattr(torch.version, "hip", None):
        target_backend = "HIP"
    elif "xla" in lower_version:
        target_backend = "XLA"
    elif "hpu" in lower_version:
        target_backend = "HPU"
    
    active_backends = ["CPU"]
    if target_backend != "CPU":
        active_backends.append(target_backend)
        
    drive_out_dir = "/content/drive/MyDrive/pytorch_aten_exports"
    os.makedirs(drive_out_dir, exist_ok=True)
    
    output_file_path = os.path.join(drive_out_dir, f"{safe_version_name}.json")
    versions_path = os.path.join(drive_out_dir, "versions.json")
    
    # Get all op names
    try:
        op_names = torch._C._dispatch_get_all_op_names()
    except AttributeError:
        op_names = []
        for ns_name in dir(torch.ops):
            ns = getattr(torch.ops, ns_name)
            for op_name in dir(ns):
                op_names.append(f"{ns_name}::{op_name}")
                
    op_names = sorted(list(set(op_names)))
    print(f"Total operators found: {len(op_names)}")
    
    records = []
    
    for op_name in op_names:
        try:
            table = torch._C._dispatch_dump_table(op_name)
            keys = parse_table_for_backends(op_name, table, active_backends)
        except Exception:
            keys = {}
            
        # 3. Filter out operators that have NO support in the active backends list
        if len(keys) > 0:
            op_data = {
                "o": op_name,
                "k": keys
            }
            records.append(op_data)
            
    # Write compressed JSON matrix
    with open(output_file_path, "w", encoding="utf-8") as f:
        json.dump(records, f)
    print(f"Successfully exported {len(records)} operators to {output_file_path}")
    
    # Update versions.json mapping
    versions_mapping = {}
    if os.path.exists(versions_path):
        try:
            with open(versions_path, "r", encoding="utf-8") as f:
                versions_mapping = json.load(f)
        except Exception:
            versions_mapping = {}
            
    # 4. Output mapping structure in versions.json
    versions_mapping[safe_version_name] = {
        "raw": torch_version,
        "backends": active_backends
    }
    
    with open(versions_path, "w", encoding="utf-8") as f:
        json.dump(versions_mapping, f, indent=2)
    print(f"Successfully updated version list mapping at: {versions_path}")

if __name__ == "__main__":
    main()


## Section 3.5: Environment Cleanup (Preventing Dependency Conflicts)

Google Colab pre-installs specific versions of `torchvision` and `torchaudio` that are strictly tied to Colab's default PyTorch version. When we downgrade or install different PyTorch versions, `pip` will trigger dependency resolver conflicts (e.g. torchvision 0.26.0+cpu requires torch==2.11.0, but you installed torch 2.8.0).

Since our ATen dispatcher exporter **only requires the core `torch` library**, we can safely uninstall `torchvision` and `torchaudio` beforehand to clean up the environment and prevent installer blocks.

In [ ]:
# Uninstall tied ecosystem libraries to prevent version conflicts when installing target PyTorch versions
!pip uninstall -y torchvision torchaudio

## Section 4: Workflow Cells

Execute the workflows below to export support matrices. 

**Important Instructions**:
1. Run the **Install PyTorch** cell.
2. Run the **Restart Runtime** cell. Wait for Google Colab to reconnect (you will see the green checkmark/connected indicator in the top right).
3. Run the **Run Exporter** cell to collect and upload data.

---
### Workflow Group: PyTorch 2.5.0

In [ ]:
!pip install -q torch==2.5.0

In [ ]:
import os
os.kill(os.getpid(), 9)

In [ ]:
!python /content/exporter.py

---
### Workflow Group: PyTorch 2.4.0

In [ ]:
!pip install -q torch==2.4.0

In [ ]:
import os
os.kill(os.getpid(), 9)

In [ ]:
!python /content/exporter.py

---
### Workflow Group: PyTorch 2.3.0

In [ ]:
!pip install -q torch==2.3.0

In [ ]:
import os
os.kill(os.getpid(), 9)

In [ ]:
!python /content/exporter.py